# Module 1 Bridge -- combined FIRE + LongDR + Tianjin caching

Replaces notebooks 04 and 05's caching cells with a single session: one Drive mount, one
GitHub token, one repo clone, one upload of the fixed `apply_to_progression_data.py`. Builds
all 3 Module 1 output caches (FIRE, LongDR, Tianjin) that notebook 06 needs for the real
combined Module 2 training run.

**Deliberately does NOT run notebook 04's own standalone Module 2 training cell** -- that was
a throwaway 5-epoch FIRE+LongDR-only POC that nothing downstream actually reads (notebook 06
trains its own Module 2 checkpoint from scratch using all 3 sources, and notebook 08 reads
06's checkpoint, not 04's). Skipping it saves a full training run for zero loss.

**After this notebook finishes:** go straight to notebook 06 (unchanged), then 08.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os

GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'main'  # <-- change if the code you need isn't merged to main yet
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'  # <-- change if you used a different folder
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
TIANJIN_ZIP = os.path.join(DRIVE_DATA_DIR, 'retinal-dr-longitudinal.zip')
assert os.path.isfile(TIANJIN_ZIP), f"Not found: {TIANJIN_ZIP}"


Mounted at /content/drive
GitHub Personal Access Token (repo scope): ··········


In [2]:
# Unzip FIRE + LongDR + Tianjin. -n skips files that already exist, so safe/cheap to re-run.
import glob, shutil

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FIRE_dataset.zip"
!unzip -q -n "$DRIVE_DATA_DIR/LongDRScreening_20150209.zip" -d LongDRScreening_20150209
!unzip -q -n "$TIANJIN_ZIP" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, 'No corrected_manifest.csv found under the extracted Tianjin zip.'
_tianjin_root = os.path.dirname(_manifest_candidates[0])
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(_tianjin_root, TIANJIN_DIR)

print('FIRE Images present :', os.path.isdir('/content/data/FIRE_dataset/FIRE/Images'))
print('LongDR norm present :', os.path.isdir('/content/data/LongDRScreening_20150209/FundusImagesNormalized'))
print('Tianjin dir present :', os.path.isdir(TIANJIN_DIR))


/content/data
FIRE Images present : True
LongDR norm present : True
Tianjin dir present : True


In [3]:
# Clone the repo once and install everything both the FIRE/LongDR and Tianjin steps need.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR}"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch opencv-python-headless


/content
Cloning into 'DRProgression'...
remote: Enumerating objects: 291, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (222/222), done.
remote: Total 291 (delta 150), reused 152 (delta 67), pack-reused 0 (from 0)
Receiving objects: 100% (291/291), 1.30 MiB | 12.25 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.8 MB/s eta 0:00:00


In [4]:
# Laterality resolution -- Tianjin-only, required before tianjin_dataset.py/apply_to_progression_data.py
# can run over it. Reuses a Drive-persisted copy if one exists (saves a few minutes).
LATERALITY_DRIVE_PATH = os.path.join(DRIVE_DATA_DIR, 'module1_cache', 'laterality_resolved.csv')
LATERALITY_LOCAL_PATH = os.path.join(TIANJIN_DIR, 'laterality_resolved.csv')

if os.path.isfile(LATERALITY_DRIVE_PATH):
    shutil.copy(LATERALITY_DRIVE_PATH, LATERALITY_LOCAL_PATH)
    print(f'✓ Reused laterality_resolved.csv from Drive')
else:
    %cd {REPO_CODE_DIR}
    !python module1/resolve_eye_laterality.py --dataset-dir "{TIANJIN_DIR}"
    os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
    shutil.copy(LATERALITY_LOCAL_PATH, LATERALITY_DRIVE_PATH)
    print(f'✓ Computed and persisted laterality_resolved.csv to Drive')


✓ Reused laterality_resolved.csv from Drive


In [5]:
# Locate Module 1 checkpoints (classifier + EX/MA + HE/SE if trained) -- shared by all 3
# apply_to_progression_data.py calls below.
import glob

SEG_DIR   = '/content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation'
CLS_SAVES = '/content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/fgadr/saves'

_pref  = ['final_weights.pt', 'best_validation_weights.pt']
_cands = [os.path.join(CLS_SAVES, n) for n in _pref] + sorted(
    glob.glob(os.path.join(CLS_SAVES, '*.pt')), key=os.path.getmtime, reverse=True)
CLS_CKPT = next((p for p in _cands if os.path.isfile(p)), None)
assert CLS_CKPT, f"No classifier .pt in {CLS_SAVES} -- run notebook 02 first"

EX_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_ex', 'model_2.pth.tar')
MA_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_ma', 'model_2.pth.tar')
for _p in (CLS_CKPT, EX_CKPT, MA_CKPT):
    assert os.path.isfile(_p), f"missing checkpoint: {_p}"

HE_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_he', 'model_2.pth.tar')
SE_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_se', 'model_2.pth.tar')
print('classifier :', CLS_CKPT)
print('EX seg     :', EX_CKPT)
print('MA seg     :', MA_CKPT)
print('HE seg     :', HE_CKPT if os.path.isfile(HE_CKPT) else 'not trained yet')
print('SE seg     :', SE_CKPT if os.path.isfile(SE_CKPT) else 'not trained yet')

_extra_seg_args = ''
if os.path.isfile(HE_CKPT):
    _extra_seg_args += f' --seg-checkpoint HE="{HE_CKPT}"'
if os.path.isfile(SE_CKPT):
    _extra_seg_args += f' --seg-checkpoint SE="{SE_CKPT}"'

os.makedirs('/content/drive/MyDrive/Thesis_Datasets/module1_cache', exist_ok=True)


classifier : /content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/fgadr/saves/final_weights.pt
EX seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation/models_FGADR_NO_TATL_ex/model_2.pth.tar
MA seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation/models_FGADR_NO_TATL_ma/model_2.pth.tar
HE seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation/models_FGADR_NO_TATL_he/model_2.pth.tar
SE seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation/models_FGADR_NO_TATL_se/model_2.pth.tar


In [10]:
# Local module1/apply_to_progression_data.py has --checkpoint-every/--resume support (added
# so a long run survives a lost Colab session or Ctrl-C) that hasn't been pushed to GitHub
# yet. Upload the current local copy once here and it covers all 3 calls below.
#
# Before running: on your machine, in the DRProgression repo, locate:
#   M2-DRProgression-VerM-module1-fgadr-poc/module1/apply_to_progression_data.py
# Run this cell, then in the file picker select that one file.
from google.colab import files

uploaded = files.upload()  # pick apply_to_progression_data.py

for fname in uploaded:
    if fname != 'apply_to_progression_data.py':
        print(f'Skipping unexpected file: {fname}')
        continue
    dest = os.path.join(REPO_CODE_DIR, 'module1', fname)
    shutil.move(fname, dest)
    print(f'Overwrote {dest}')

!python {REPO_CODE_DIR}/module1/apply_to_progression_data.py --help | grep -E "checkpoint-every|resume"


Saving apply_to_progression_data.py to apply_to_progression_data (1).py
Skipping unexpected file: apply_to_progression_data (1).py
                                    [--checkpoint-every CHECKPOINT_EVERY]
                                    [--resume]
  --checkpoint-every CHECKPOINT_EVERY
  --resume              If --out already exists, load it first and skip


In [14]:
!sed -i 's/cache = torch.load(args.out)/cache = torch.load(args.out, weights_only=False)/' /content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module1/apply_to_progression_data.py


In [15]:
# FIRE
%cd {REPO_CODE_DIR}/module1
!python apply_to_progression_data.py \
    --images-dir /content/data/FIRE_dataset/FIRE/Images \
    --classifier-checkpoint "{CLS_CKPT}" \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --checkpoint-every 50 --resume \
    --out /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_fire.pt


/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module1
↻ resuming: 268 entries already in /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_fire.pt, skipping those image_ids
✓ wrote 268 entries to /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_fire.pt


In [16]:
# LongDR
!python apply_to_progression_data.py \
    --images-dir /content/data/LongDRScreening_20150209/FundusImagesNormalized \
    --classifier-checkpoint "{CLS_CKPT}" \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --checkpoint-every 50 --resume \
    --out /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_longdr.pt


↻ resuming: 1120 entries already in /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_longdr.pt, skipping those image_ids
✓ wrote 1120 entries to /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_longdr.pt


In [18]:
import os
p = '/content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_tianjin.pt'
print(os.path.getsize(p), 'bytes')


0 bytes


In [19]:
os.remove(p)


In [20]:
# Tianjin -- the slow one. If this gets interrupted, just re-run this exact cell: --resume
# means it picks up where it left off instead of starting over.
!python apply_to_progression_data.py \
    --images-dir "{TIANJIN_DIR}/baseline fundus images" \
    --classifier-checkpoint "{CLS_CKPT}" \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --checkpoint-every 50 --resume \
    --out /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_tianjin.pt


⚠ 00194__._00194-7254: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7254.jpg')
⚠ 00194__._00194-7256: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7256.jpg')
⚠ 00194__._00194-7258: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7258.jpg')
⚠ 00194__._00194-7265: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7265.jpg')
✓ 00194__00194-7256: grade=3 lbs=0.00000 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ 00194__00194-7261: grade=3 lbs=0.00000 (lesions: ['EX', 'HE', 'MA', 'SE'])
⚠ 00197__._00197-7299: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00197/._00197-7299.jpg')
⚠ 00197__._00197-7302: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/0

## Done

All 3 caches are now on Drive under `module1_cache/`. Next: run notebook 06
(`06_module2_poc_with_tianjin_colab.ipynb`) unchanged -- it reads these same cache paths and
trains the real combined Module 2 checkpoint. Then notebook 08 for trajectory visualization.